In [1]:
# Import required libraries
import pandas as pd
import requests
import json

In [2]:
# get apikey from json file

# Load API key from JSON file
with open('api_data_gov.json') as f:
    api_key = json.load(f)['api_key']

In [3]:
# Step 1: Set up API call and fetch data
url = "https://api.data.gov/ed/collegescorecard/v1/schools"
params = {
    'api_key': api_key,
    'per_page': 200,
    'school.degrees_awarded.predominant': 2
}

In [4]:

# Step 1: Set up API call and fetch data
url = "https://api.data.gov/ed/collegescorecard/v1/schools?api_key=mPP6fSIgoZPew6veCzpsQ32mKxn4pRMtcDgjUBkr&per_page=200&school.degrees_awarded.predominant=2"
try:
    # Fetch data from API
    response = requests.get(url, params=params)
    response.raise_for_status()  # This will raise an exception for bad status codes
    
    # Print response status and headers for debugging
    print(f"Response Status Code: {response.status_code}")
    # print(f"Response Headers: {response.headers}")
    
    data = response.json()
except requests.exceptions.RequestException as e:
    print(f"API Error: {e}")
except Exception as e:
    print(f"Processing Error: {e}")

# df = pd.DataFrame(data['results'])

Response Status Code: 200


In [5]:
# lets normalize the JSON data into a flat table
df = pd.json_normalize(data['results']) 
df.head()

,id,ope6_id,ope8_id,fed_sch_cd,latest.school.zip,latest.school.city,latest.school.name,latest.school.alias,latest.school.state,latest.school.locale,...,school.title_iv.approval_date,school.title_iv.eligibility_type,school.degrees_awarded.highest,school.degrees_awarded.predominant,school.degrees_awarded.predominant_recoded,school.tuition_revenue_per_fte,school.instructional_expenditure_per_fte,school.institutional_characteristics.level,location.lat,location.lon
0,101161,001060,00106000,001060,36507-2698,Bay Minette,Coastal Alabama Community College,None,AL,32,...,6/14/1985,1,2,2,2.0,2251,6906,2,30.851343,-87.778193
1,101301,005699,00569900,009980,36702-2808,Selma,George C Wallace State Community College-Selma,Wallace Community College Selma,AL,32,...,8/7/1980,1,2,2,2.0,2491,6918,2,32.445917,-87.013294
2,101505,001022,00102200,001022,35215-3098,Birmingham,Jefferson State Community College,Jeff State | JSCC | Jefferson State,AL,12,...,7/3/1968,1,2,2,2.0,3166,5831,2,33.652535,-86.707233
3,101569,001059,00105900,001059,35221-1717,Birmingham,Lawson State Community College,None,AL,12,...,5/1/1967,1,2,2,1.0,1121,8826,2,33.450814,-86.888844
4,101648,001026,00102600,001026,36756-3207,Marion,Marion Military Institute,None,AL,43,...,3/1/1970,1,2,2,2.0,2711,5927,2,32.623734,-87.322041


In [6]:
# 1 lets rename the columns to make them more readable. 
# This will also make them consistent with the other sourced datasets  

df.rename(columns={
    'id': 'school_id',
    'latest.school.name': 'school_name',
    'latest.school.city': 'city',
    'latest.school.state': 'state',
    'latest.school.zip': 'zip_code',
    'latest.admissions.admission_rate.overall': 'admission_rate',
    'latest.cost.tuition.in_state': 'tuition_in_state',
    'latest.cost.tuition.out_of_state': 'tuition_out_of_state',
    'latest.cost.attendance.academic_year': 'cost_attendance_academic_year',
    'latest.student.size': 'student_size',
    'latest.student.demographics.men': 'men',
    'latest.student.demographics.women': 'women',
    'latest.student.demographics.unemployment': 'unemployment',
    'latest.student.demographics.age_entry': 'age',
    'latest.student.demographics.married': 'married',
    'latest.student.demographics.unemployment': 'unemployment',
    'latest.student.demographics.female_share': 'female_share',
    'latest.student.demographics.median_hh_income': 'median_hh_income',
    },  inplace=True)
df.head()

,school_id,ope6_id,ope8_id,fed_sch_cd,zip_code,city,school_name,latest.school.alias,state,latest.school.locale,...,school.title_iv.approval_date,school.title_iv.eligibility_type,school.degrees_awarded.highest,school.degrees_awarded.predominant,school.degrees_awarded.predominant_recoded,school.tuition_revenue_per_fte,school.instructional_expenditure_per_fte,school.institutional_characteristics.level,location.lat,location.lon
0,101161,001060,00106000,001060,36507-2698,Bay Minette,Coastal Alabama Community College,None,AL,32,...,6/14/1985,1,2,2,2.0,2251,6906,2,30.851343,-87.778193
1,101301,005699,00569900,009980,36702-2808,Selma,George C Wallace State Community College-Selma,Wallace Community College Selma,AL,32,...,8/7/1980,1,2,2,2.0,2491,6918,2,32.445917,-87.013294
2,101505,001022,00102200,001022,35215-3098,Birmingham,Jefferson State Community College,Jeff State | JSCC | Jefferson State,AL,12,...,7/3/1968,1,2,2,2.0,3166,5831,2,33.652535,-86.707233
3,101569,001059,00105900,001059,35221-1717,Birmingham,Lawson State Community College,None,AL,12,...,5/1/1967,1,2,2,1.0,1121,8826,2,33.450814,-86.888844
4,101648,001026,00102600,001026,36756-3207,Marion,Marion Military Institute,None,AL,43,...,3/1/1970,1,2,2,2.0,2711,5927,2,32.623734,-87.322041


In [7]:
# 2 lets keep only the columns we need and we just renamed.

df = df[['school_id', 'school_name', 'city', 'state', 'zip_code', 'admission_rate', 'tuition_in_state', 'tuition_out_of_state', 'cost_attendance_academic_year', 'student_size', 'men', 'women', 'unemployment', 'age', 'married', 'female_share', 'median_hh_income']]

df.head()

,school_id,school_name,city,state,zip_code,admission_rate,tuition_in_state,tuition_out_of_state,cost_attendance_academic_year,student_size,men,women,unemployment,age,married,female_share,median_hh_income
0,101161,Coastal Alabama Community College,Bay Minette,AL,36507-2698,NaN,4980.0,8790.0,15171.0,4714,0.3432,0.6568,3.44,23.0,0.095528,0.653179,52162.0
1,101301,George C Wallace State Community College-Selma,Selma,AL,36702-2808,NaN,4680.0,8490.0,9505.0,828,0.3635,0.6365,5.14,22.0,0.053680,0.599134,35579.0
2,101505,Jefferson State Community College,Birmingham,AL,35215-3098,NaN,5040.0,8850.0,14384.0,5443,0.3489,0.6511,3.15,24.0,0.133796,0.660555,57833.0
3,101569,Lawson State Community College,Birmingham,AL,35221-1717,NaN,4980.0,8790.0,15028.0,2720,0.3467,0.6533,5.31,23.0,0.049759,0.568218,40437.0
4,101648,Marion Military Institute,Marion,AL,36756-3207,0.5917,9538.0,15538.0,18458.0,308,0.7305,0.2695,3.58,19.0,NaN,0.234043,56305.0


In [8]:
# Find missings and fill forward

# find the sum of missings in columns 


print(f"missings before fillforward:\n {df.isnull().sum()} \n \n ")


# and fill forward the missing values
df = df.fillna(method='ffill')

print(f"missings after fillforward:\n {df.isnull().sum()}\n \n")

missings before fillforward:
 school_id                         0
school_name                       0
city                              0
state                             0
zip_code                          0
admission_rate                   93
tuition_in_state                  6
tuition_out_of_state              6
cost_attendance_academic_year     7
student_size                      0
men                               0
women                             0
unemployment                      3
age                               2
married                           4
female_share                      3
median_hh_income                  3
dtype: int64 
 
 
missings after fillforward:
 school_id                        0
school_name                      0
city                             0
state                            0
zip_code                         0
admission_rate                   4
tuition_in_state                 0
tuition_out_of_state             0
cost_attendance_academic_year  

/var/folders/cv/71w18t8n27b_81x9rhwkm1080000gq/T/ipykernel_80905/3716669808.py:10: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='ffill')


In [9]:
# 4 find the outliers in the data & remove them 

def remove_outliers(df, column):
    # Calculate the first and third quartiles
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    # Calculate the interquartile range (IQR)
    IQR = Q3 - Q1

    # Determine the lower and upper bounds for outliers
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Remove outliers
    df = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

    return df

print(f" Shape before removing outliers: {df.shape}")
#  Create a for loop to iterate over the columns and remove outliers
columns_to_check = ['admission_rate', 'tuition_in_state', 'tuition_out_of_state', 'cost_attendance_academic_year', 'student_size', 'men', 'women', 'unemployment', 'age', 'married', 'female_share', 'median_hh_income']

for column in columns_to_check:
    df = remove_outliers(df, column)

print(f" Shape after removing outliers: {df.shape}")

 Shape before removing outliers: (100, 17)
 Shape after removing outliers: (57, 17)


In [10]:
#5 format columns which indicates percentages to percentages instead of floats ( men, women, married & female_share)

df['men'] = df['men'].apply(lambda x: x * 100)
df['women'] = df['women'].apply(lambda x: x * 100)
df['married'] = df['married'].apply(lambda x: x * 100)
df['female_share'] = df['female_share'].apply(lambda x: x * 100)

df.head()

,school_id,school_name,city,state,zip_code,admission_rate,tuition_in_state,tuition_out_of_state,cost_attendance_academic_year,student_size,men,women,unemployment,age,married,female_share,median_hh_income
16,108667,College of Alameda,Alameda,CA,94501,1.0000,1148.0,8684.0,17578.0,4274,42.40,57.60,4.40,25.0,14.370370,54.296296,61586.0
17,108807,Allan Hancock College,Santa Maria,CA,93454-6399,1.0000,1156.0,9436.0,11544.0,7812,46.54,53.46,4.60,23.0,10.098432,53.700328,56468.0
19,109208,American River College,Sacramento,CA,95841-4286,0.8968,1288.0,12808.0,12986.0,25758,53.63,46.37,4.41,26.0,20.086160,57.444803,59196.0
20,109350,Antelope Valley Community College District,Lancaster,CA,93536-5426,0.8968,1124.0,8228.0,15158.0,10905,43.13,56.87,6.35,24.0,8.785656,58.859006,58643.0
23,110246,Butte College,Oroville,CA,95965-8399,0.8968,1336.0,8416.0,13636.0,8384,43.99,56.01,5.39,23.0,9.027336,51.260860,46805.0


In [11]:
#6 Find duplicate rows if any and remove them
duplicates = df.duplicated()
print(f"Number of duplicate rows: {duplicates.sum()}")
if duplicates.sum() > 0:
    df = df[~duplicates]
    print(f"Duplicate rows removed. New shape: {df.shape}")
else:
    print("No duplicate rows found.")

df.head()

Number of duplicate rows: 0
No duplicate rows found.


,school_id,school_name,city,state,zip_code,admission_rate,tuition_in_state,tuition_out_of_state,cost_attendance_academic_year,student_size,men,women,unemployment,age,married,female_share,median_hh_income
16,108667,College of Alameda,Alameda,CA,94501,1.0000,1148.0,8684.0,17578.0,4274,42.40,57.60,4.40,25.0,14.370370,54.296296,61586.0
17,108807,Allan Hancock College,Santa Maria,CA,93454-6399,1.0000,1156.0,9436.0,11544.0,7812,46.54,53.46,4.60,23.0,10.098432,53.700328,56468.0
19,109208,American River College,Sacramento,CA,95841-4286,0.8968,1288.0,12808.0,12986.0,25758,53.63,46.37,4.41,26.0,20.086160,57.444803,59196.0
20,109350,Antelope Valley Community College District,Lancaster,CA,93536-5426,0.8968,1124.0,8228.0,15158.0,10905,43.13,56.87,6.35,24.0,8.785656,58.859006,58643.0
23,110246,Butte College,Oroville,CA,95965-8399,0.8968,1336.0,8416.0,13636.0,8384,43.99,56.01,5.39,23.0,9.027336,51.260860,46805.0


# 1 paragraph of the ethical implications of data wrangling specific to your datasource and the steps you completed answering the following questions:
1.  What changes were made to the data?
2.  Are there any legal or regulatory guidelines for your data or project topic?
3.  What risks could be created based on the transformations done?
4.  Did you make any assumptions in cleaning/transforming the data?
5.  How was your data sourced / verified for credibility?
6.  Was your data acquired in an ethical way?
7.  How would you mitigate any of the ethical implications you have identified?




The College Scorecard data transformation process involved several significant changes  including column renaming, missing value imputation, outlier removal, and percentage standardization. These modifications fall under FERPA (Family Educational Rights and Privacy Act) guidelines & educational data usage as mentioned on the website. This information is asked to use responsibly under the guidelines as mentioned above. Key risks emerged from our transformations could include forward filling the imputation could propagate incorrect values, IQR based outlier removal might exclude valid but unusual institutions, and percentage standardization assumes linear relationships. We made critical assumptions during cleaning, particularly that missing values followed patterns suitable for forward fill and that outliers represented errors rather than valid extreme cases. The data was sourced ethically through the official Department of Education API with proper authentication, ensuring credibility and compliance with federal guidelines using the key provided by the website so no ethical concerns about it. To mitigate  ethical concerns, i would recomment recommend; maintaining original values alongside transformed data and documenting all transformation decisions. Inncluding confidence intervals for imputed values could also make things more transparent and at the same time, regularly validating assumptions against new data will make sure you are keeping up  with the updates.
